# Phase 07 — Dense + BM25 hybrid retrieval

This notebook compares three candidate-generation paths:

```text
Business question ──┬── BAAI/bge-m3 query embedding → Qdrant cosine dense search ──┐
                    └── tokenization → BM25 sparse search ───────────────────────────┤
                                                                                         ↓
                                                         reciprocal-rank fusion → final candidates
```

BM25 is directly executable over the Phase 03 chunk corpus. Dense and hybrid results execute only if Phase 04 has produced **real** `BAAI/bge-m3` embeddings. The notebook does not use a substitute embedding model, fake vectors, fake scores, or fake dense/hybrid results.

> **Phase boundary:** no reranking and no LangGraph are implemented here.

In [1]:
from __future__ import annotations

import json
import re
import uuid
from collections import defaultdict
from pathlib import Path
from typing import Any

from rank_bm25 import BM25Okapi

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CHUNKS_PATH = PROCESSED_DIR / "introduction_to_business_chunks.jsonl"
EMBEDDINGS_PATH = PROCESSED_DIR / "introduction_to_business_bge_m3_embeddings.jsonl"
EMBEDDING_STATUS_PATH = PROCESSED_DIR / "introduction_to_business_bge_m3_embedding_status.json"
QDRANT_STATUS_PATH = PROCESSED_DIR / "introduction_to_business_qdrant_indexing_status.json"
HYBRID_STATUS_PATH = PROCESSED_DIR / "introduction_to_business_hybrid_retrieval_status.json"
BM25_RESULTS_PATH = PROCESSED_DIR / "introduction_to_business_bm25_retrieval_results.json"
HYBRID_RESULTS_PATH = PROCESSED_DIR / "introduction_to_business_hybrid_retrieval_results.json"

MODEL_NAME = "BAAI/bge-m3"
COLLECTION_NAME = "openstax_introduction_to_business_bge_m3_hybrid"
TOP_K_VALUES = (3, 5, 8)
RRF_CONSTANT = 60
REQUIRED_FIELDS = ("chunk_id", "text", "source", "page", "chapter", "section")

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

print(f"Project root: {PROJECT_ROOT}")
print(f"Chunk corpus: {CHUNKS_PATH}")

Project root: /home/ubuntu/business-knowledge-ai
Chunk corpus: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_chunks.jsonl


## Test questions, Top-K experiments, and comparison rules

The same questions are sent to every enabled retrieval path. For each value of K, inspect topical relevance, coverage, citation metadata, overlap between candidate sets, and whether fusion adds useful evidence rather than duplicates. Reciprocal-rank fusion (RRF) combines rank signals only; it is not a reranker.

In [2]:
SAMPLE_QUESTIONS = [
    {"query_id": "business-foundations", "question": "What role do businesses play in an economy?"},
    {"query_id": "ownership-choice", "question": "What factors should an entrepreneur consider when choosing a business ownership structure?"},
    {"query_id": "management-planning", "question": "How does the planning process help managers set organizational goals?"},
    {"query_id": "marketing-selling", "question": "How is marketing different from selling?"},
    {"query_id": "financial-statements", "question": "How do financial statements help business owners make decisions?"},
]

MANUAL_REVIEW_CRITERIA = (
    "For every candidate set, check topical relevance, coverage of the question, usable page/chapter/section citation metadata, "
    "and whether the added candidates are distinct rather than redundant."
)
print(f"Questions: {len(SAMPLE_QUESTIONS)} | K values: {TOP_K_VALUES} | RRF constant: {RRF_CONSTANT}")

Questions: 5 | K values: (3, 5, 8) | RRF constant: 60


In [3]:
chunk_records = [
    json.loads(line)
    for line in CHUNKS_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
if not chunk_records:
    raise ValueError("The Phase 03 chunk artifact is empty.")
if any(any(field not in record for field in REQUIRED_FIELDS) for record in chunk_records):
    raise ValueError("A chunk record is missing required provenance metadata.")

corpus_tokens = [tokenize(record["text"]) for record in chunk_records]
if any(not tokens for tokens in corpus_tokens):
    raise ValueError("A chunk tokenized to an empty sparse document.")
bm25 = BM25Okapi(corpus_tokens)

def base_result(record: dict[str, Any]) -> dict[str, Any]:
    return {field: record[field] for field in REQUIRED_FIELDS}

def bm25_search(question: str, top_k: int) -> list[dict[str, Any]]:
    import numpy as np
    scores = bm25.get_scores(tokenize(question))
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [
        {**base_result(chunk_records[int(index)]), "rank": rank, "bm25_score": float(scores[int(index)])}
        for rank, index in enumerate(top_indices, start=1)
    ]

print(f"BM25 corpus is ready with {len(chunk_records):,} provenance-preserving chunks.")

BM25 corpus is ready with 3,018 provenance-preserving chunks.


In [4]:
embedding_status = json.loads(EMBEDDING_STATUS_PATH.read_text(encoding="utf-8"))
qdrant_status = json.loads(QDRANT_STATUS_PATH.read_text(encoding="utf-8"))
embedding_artifact_exists = EMBEDDINGS_PATH.is_file() and EMBEDDINGS_PATH.stat().st_size > 0
DENSE_READY = bool(embedding_status.get("embeddings_generated")) and embedding_artifact_exists

preflight = {
    "phase": "07_hybrid_retrieval",
    "dense_model": MODEL_NAME,
    "embedding_status": embedding_status.get("status"),
    "qdrant_status": qdrant_status.get("status"),
    "embedding_artifact_exists": embedding_artifact_exists,
    "dense_retrieval_ready": DENSE_READY,
    "bm25_ready": True,
    "fusion_strategy": "reciprocal_rank_fusion",
    "rrf_constant": RRF_CONSTANT,
    "reranking_implemented": False,
    "langgraph_implemented": False,
    "no_model_substitution": True,
    "no_fabricated_dense_scores_or_hybrid_results": True,
}

if DENSE_READY:
    print("Dense, BM25, and hybrid execution paths are eligible.")
else:
    print("Dense and hybrid paths are blocked: Phase 04 lacks a real BGE-M3 artifact. BM25 remains executable.")

Dense and hybrid paths are blocked: Phase 04 lacks a real BGE-M3 artifact. BM25 remains executable.


## Exact dense path and deterministic fusion

When the real embedding artifact exists, this cell reconstructs an ephemeral cosine Qdrant collection from it, generates BGE-M3 query vectors, and executes dense retrieval. RRF sums `1 / (60 + rank)` across dense and BM25 rank lists; it does not inspect text semantics or reorder candidates with a learned model.

In [5]:
def reciprocal_rank_fusion(named_rank_lists: dict[str, list[dict[str, Any]]], rrf_constant: int = RRF_CONSTANT) -> list[dict[str, Any]]:
    fused: dict[str, dict[str, Any]] = {}
    for retriever_name, results in named_rank_lists.items():
        for result in results:
            chunk_id = result["chunk_id"]
            candidate = fused.setdefault(
                chunk_id,
                {**{field: result[field] for field in REQUIRED_FIELDS}, "retriever_ranks": {}, "fused_score": 0.0},
            )
            rank = int(result["rank"])
            candidate["retriever_ranks"][retriever_name] = rank
            candidate["fused_score"] += 1.0 / (rrf_constant + rank)
    ranked = sorted(fused.values(), key=lambda item: (-item["fused_score"], item["chunk_id"]))
    return [{**item, "rank": rank} for rank, item in enumerate(ranked, start=1)]

if DENSE_READY:
    import numpy as np
    from FlagEmbedding import BGEM3FlagModel
    from qdrant_client import QdrantClient, models

    embedding_records = [json.loads(line) for line in EMBEDDINGS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    if len(embedding_records) != len(chunk_records):
        raise ValueError("Embedding and chunk records are not aligned.")

    def vector_from(record: dict[str, Any]) -> list[float]:
        vector = record.get("embedding", record.get("dense_embedding"))
        if not isinstance(vector, list) or not vector:
            raise ValueError("Missing a real dense vector.")
        return [float(value) for value in vector]

    vector_dimension = len(vector_from(embedding_records[0]))
    if vector_dimension != 1024 or any(len(vector_from(item)) != vector_dimension for item in embedding_records):
        raise ValueError("Exact BGE-M3 1024-dimensional vectors are required.")

    qdrant_client = QdrantClient(":memory:")
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(size=vector_dimension, distance=models.Distance.COSINE),
    )
    qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=[
            models.PointStruct(
                id=str(uuid.uuid5(uuid.NAMESPACE_URL, item["chunk_id"])),
                vector=vector_from(item),
                payload={field: item[field] for field in REQUIRED_FIELDS},
            )
            for item in embedding_records
        ],
        wait=True,
    )
    assert qdrant_client.count(COLLECTION_NAME, exact=True).count == len(embedding_records)

    query_model = BGEM3FlagModel(MODEL_NAME, use_fp16=True)
    raw_query_vectors = np.asarray(query_model.encode([item["question"] for item in SAMPLE_QUESTIONS], batch_size=4)["dense_vecs"], dtype=np.float32)
    query_vectors = raw_query_vectors / np.linalg.norm(raw_query_vectors, axis=1, keepdims=True)
    if query_vectors.shape != (len(SAMPLE_QUESTIONS), 1024):
        raise ValueError("Unexpected BGE-M3 query vector shape.")

    def dense_search(query_vector: np.ndarray, top_k: int) -> list[dict[str, Any]]:
        response = qdrant_client.query_points(COLLECTION_NAME, query=query_vector.tolist(), limit=top_k, with_payload=True, with_vectors=False)
        return [
            {**{field: point.payload[field] for field in REQUIRED_FIELDS}, "rank": rank, "dense_score": float(point.score)}
            for rank, point in enumerate(response.points, start=1)
        ]
else:
    query_vectors = []
    print("Exact dense search and reciprocal-rank fusion are intentionally unavailable until real BGE-M3 vectors exist.")

Exact dense search and reciprocal-rank fusion are intentionally unavailable until real BGE-M3 vectors exist.


In [6]:
bm25_runs: list[dict[str, Any]] = []
dense_runs: list[dict[str, Any]] = []
hybrid_runs: list[dict[str, Any]] = []

for question_index, question in enumerate(SAMPLE_QUESTIONS):
    for top_k in TOP_K_VALUES:
        sparse_results = bm25_search(question["question"], top_k)
        bm25_runs.append({"query_id": question["query_id"], "question": question["question"], "top_k": top_k, "results": sparse_results})
        if DENSE_READY:
            dense_results = dense_search(query_vectors[question_index], top_k)
            dense_runs.append({"query_id": question["query_id"], "question": question["question"], "top_k": top_k, "results": dense_results})
            final_candidates = reciprocal_rank_fusion({"dense": dense_results, "bm25": sparse_results})[:top_k]
            hybrid_runs.append({"query_id": question["query_id"], "question": question["question"], "top_k": top_k, "results": final_candidates})

assert len(bm25_runs) == len(SAMPLE_QUESTIONS) * len(TOP_K_VALUES)
if DENSE_READY:
    assert len(dense_runs) == len(bm25_runs) == len(hybrid_runs)
print(f"BM25 runs completed: {len(bm25_runs)}")
print(f"Dense runs completed: {len(dense_runs)} | Hybrid runs completed: {len(hybrid_runs)}")

BM25 runs completed: 15
Dense runs completed: 0 | Hybrid runs completed: 0


In [7]:
def print_result_set(label: str, results: list[dict[str, Any]], score_field: str) -> None:
    print(f"\n{label}")
    for result in results:
        preview = result["text"].replace("\n", " ")[:360]
        print(
            f"#{result['rank']} | {score_field}={result[score_field]:.4f} | page={result['page']} | "
            f"chapter={result['chapter']} | section={result['section']}\n"
            f"source={result['source']}\n{preview}\n"
        )

for question in SAMPLE_QUESTIONS:
    selected_k = 5
    sparse_run = next(run for run in bm25_runs if run["query_id"] == question["query_id"] and run["top_k"] == selected_k)
    print("=" * 108)
    print(f"Question: {question['question']} | Top-K: {selected_k}")
    print_result_set("BM25 only (executed)", sparse_run["results"], "bm25_score")
    if DENSE_READY:
        dense_run = next(run for run in dense_runs if run["query_id"] == question["query_id"] and run["top_k"] == selected_k)
        hybrid_run = next(run for run in hybrid_runs if run["query_id"] == question["query_id"] and run["top_k"] == selected_k)
        print_result_set("Dense only (executed)", dense_run["results"], "dense_score")
        print_result_set("Hybrid RRF final candidates (executed)", hybrid_run["results"], "fused_score")
    else:
        print("\nDense only and hybrid: not executed — no real BGE-M3 artifact exists.")

print("Manual comparison criterion:", MANUAL_REVIEW_CRITERIA)

Question: What role do businesses play in an economy? | Top-K: 5

BM25 only (executed)
#1 | bm25_score=20.1628 | page=193 | chapter={'number': 5, 'title': 'Entrepreneurship: Starting and Managing Your Own Business'} | section={'number': '5.4', 'title': 'provides a more detailed look at small-business owners.'}
source={'title': 'Introduction to Business', 'publisher': 'OpenStax, Rice University', 'official_book_url': 'https://openstax.org/details/books/introduction-business', 'official_pdf_url': 'https://assets.openstax.org/oscms-prodcms/media/documents/IntroductionToBusiness-OP_8D04gAa.pdf', 'local_filename': 'data/raw/introduction-to-business-openstax.pdf'}
Chapter 5 Entrepreneurship: Starting and Managing Your Own Business 181 C O N C E P T C H E C K 1. Describe the personality traits and skills characteristic of successful entrepreneurs. 2. What does it mean when we say that an entrepreneur should work on the business, not in it? 5.3 Small Business: Driving America's Growth 3. How d

In [8]:
write_json(BM25_RESULTS_PATH, bm25_runs)

status = {
    **preflight,
    "bm25": {
        "status": "completed_on_real_phase_03_chunks",
        "corpus_chunk_count": len(chunk_records),
        "results_artifact": str(BM25_RESULTS_PATH.relative_to(PROJECT_ROOT)),
        "run_count": len(bm25_runs),
    },
    "dense": {
        "status": "completed" if DENSE_READY else "blocked_missing_real_bge_m3_embeddings",
        "run_count": len(dense_runs),
    },
    "hybrid": {
        "status": "completed" if DENSE_READY else "blocked_missing_real_bge_m3_embeddings",
        "run_count": len(hybrid_runs),
        "results_artifact": str(HYBRID_RESULTS_PATH.relative_to(PROJECT_ROOT)) if DENSE_READY else None,
    },
    "manual_comparison_required": True,
    "reranking_implemented": False,
    "langgraph_implemented": False,
}
if DENSE_READY:
    write_json(HYBRID_RESULTS_PATH, {"dense_runs": dense_runs, "hybrid_runs": hybrid_runs})
else:
    status["limitation"] = (
        "BM25 was executed over real Phase 03 chunks, but exact BGE-M3 dense and fused hybrid retrieval are blocked because "
        "Phase 04 has not produced real embeddings. No dense scores or hybrid candidates were fabricated."
    )
write_json(HYBRID_STATUS_PATH, status)
print(f"Saved BM25 results: {BM25_RESULTS_PATH}")
print(f"Saved hybrid status: {HYBRID_STATUS_PATH}")

Saved BM25 results: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_bm25_retrieval_results.json
Saved hybrid status: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_hybrid_retrieval_status.json


## Phase boundary

This phase implements sparse BM25 retrieval, the exact BGE-M3/Qdrant dense path, and deterministic reciprocal-rank fusion. It does **not** implement reranking, answer generation, LLM calls, LangGraph, query rewriting, or any agentic control flow.